In [1]:
import numpy as np
from scipy import optimize
import scipy as sp

import matplotlib.pyplot as plt
import matplotlib.axes as axe
import pandas as pd
import datetime as dt
import gurobipy as gp
from gurobipy import GRB
import cvxpy as cp
import yaml

import random
from itertools import chain, combinations, tee
import time

import glob
import os


plt.rcParams['text.usetex'] = True

# Functions

In [ ]:
# def poly(alpha, x, p=4):
#     assert alpha.shape[0] == p+1,  "alpha should be a length-(p+1) vector."
#     assert np.all(alpha > 0.0), "All entries of alpha should be strictly positive."
#     assert x >= 0.0, "x should be non-negative."
    
#     return sum(alpha[k] * (x**k) for k in range(p))

# def poly_simplified(alpha, x, p=4):
#     assert alpha.shape[0] == p+1,  "alpha should be a length-(p+1) vector."
#     assert np.all(alpha > 0.0), "All entries of alpha should be strictly positive."
#     assert x >= 0.0, "x should be non-negative."
    
#     return alpha[0] + alpha[p-1] * (x**(p-1))

# def route_latency(alphas, xs, edges, p=4):
#     assert len(alphas.shape) == 2, "alphas should be a 2D array."
#     assert alphas.shape[0] == xs.shape[0], "The first dimension of alphas should equal the dimension of xs."
#     assert set(edges).issubset(range(alphas.shape[0])), "edges should be a subset of indices of edges"
    
#     return sum([poly(alphas[e, :], xs[e], p=4) for e in edges])


In [104]:
# DR: distributionally robust

def generate_R_from_edge_node_matrix(edge_node_matrix):
    assert len(edge_node_matrix.shape) == 2
    assert edge_node_matrix.shape[1] == 2
    
    num_edges = edge_node_matrix.shape[0]
    num_nodes = np.max(edge_node_matrix) + 1
    
    R = np.zeros((num_nodes - 1, num_edges))
    
    for node_index in range(R.shape[0]):
        for edge_index in range(R.shape[1]):
            if edge_node_matrix[edge_index][0] == node_index:
                R[node_index, edge_index] = 1
            elif edge_node_matrix[edge_index][1] == node_index:
                R[node_index, edge_index] = -1

    return R


def generate_Gamma(B, R):

    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    
    return inv_B - inv_B @ R.T @ R_inv_B_R_top @ R @ inv_B


def compute_eps_max(Gamma, B, R, alpha_mean_nominal, eta, delta):
    
    print()
    print("Gamma.shape", Gamma.shape)
    print("alpha_mean_nominal.shape:", alpha_mean_nominal.shape)
    print()
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec

    # Define and solve the CVXPY problem.
    eps = cp.Variable(1)
    tau = cp.Variable(num_edges)
    func = eps
    obj = cp.Maximize(func)
    constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
                 - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr += [tau >= 0.0]
    constr += [eps >= 0.0]
    prob = cp.Problem(obj, constr)
    prob.solve()

    # Print result.
    print()
    print("eps_max:", eps.value[0])
    print()

    return eps.value
    

def compute_opt_DR_toll(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec
    
    tau = cp.Variable(num_edges)
    func = eps * cp.norm(Gamma @ tau + offset_with_eta) \
            + cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
#     func = cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
    obj = cp.Minimize(func)
    constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
                 - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr += [tau >= 0.0]
    
    prob = cp.Problem(obj, constr)
    prob.solve()

    # Print result.
    print()
    print("DR optimal toll:", tau.value)
    print()
    print("Worst-case DR latency:", prob.value)
    print()
    
    return tau.value, prob.value


# Sioux Falls Network

In [105]:
num_nodes = 7
num_edges = 10

In [106]:
offsets_orig = np.array([1.33, 1.20, 2.77, 1.48, 1.00, 0.70, 0.89, 1.34, 1.91, 1.61])
quartic_coefficients_orig = np.array([0.02, 0.02, 0.03, 0.03, 0.01, 0.02, 0.01, 0.04, 0.03, 0.04])
num_travelers_orig = 15

# Version 1:

num_travelers_to_use_for_offset = 3
num_travelers_to_use = 9
beta = np.zeros(num_edges)
alpha_mean_nominal = [offsets_orig[e] + quartic_coefficients_orig[e] * num_travelers_to_use_for_offset**4 for e in range(num_edges)]

for e in range(num_edges):
    traveler_count_LS = np.array(list(range(num_travelers_to_use+1)))
    latency_data_LS = np.array([offsets_orig[e] + quartic_coefficients_orig[e] * (k**4) for k in traveler_count_LS])
    beta[e] = np.dot(traveler_count_LS, latency_data_LS - alpha_mean_nominal) / np.dot(traveler_count_LS, traveler_count_LS)
    # print(beta[e])

alpha_mean_nominal = np.array(alpha_mean_nominal)
    
print("beta:\n", beta)
print()
print("alpha_mean_nominal:\n", alpha_mean_nominal)

## Version 2: Affine Regression:

# num_travelers_to_use = 5
# beta = np.zeros(num_edges)
# alpha_mean_nominal = np.zeros(num_edges)

# for e in range(num_edges):
#     traveler_count_LS = np.array(list(range(num_travelers_to_use+1)))
#     latency_data_LS = np.array([offsets_orig[e] + quartic_coefficients_orig[e] * (k**4) for k in traveler_count_LS])
#     data_matrix_LS = np.vstack([traveler_count_LS, np.ones(len(traveler_count_LS))]).T
#     beta[e], alpha_mean_nominal[e] = np.linalg.lstsq(data_matrix_LS, latency_data_LS, rcond=None)[0]

# print(beta)
# print()
# print(alpha_mean_nominal)

## Version 3: Straight line connecting points:

# assert alpha.shape[0] == num_edges
# assert quartic_coefficients.shape[0] == num_edges
# # Maximum num_players_affine_approx = 15
# num_players_affine_approx = 10
# beta = np.array([quartic_coefficients_orig[i] * num_players_affine_approx**(4-1) for i in range(num_edges)])
# alpha_mean_nominal[e] = offsets_orig[e]
# print(beta)


beta:
 [ 8.11203509  8.09150877 12.57887719 12.37519298  3.82045614  8.0125614
  3.80308772 16.5925614  12.44308772 16.63519298]

alpha_mean_nominal:
 [2.95 2.82 5.2  3.91 1.81 2.32 1.7  4.58 4.34 4.85]


In [107]:
# e = 0

# traveler_count_line = np.linspace(np.min(traveler_count_LS), np.max(traveler_count_LS), 100)
# latency_line = beta[e] * traveler_count_line + alpha_mean_nominal[e]

# plt.scatter(traveler_count_LS, latency_data_LS, color='blue', label='Data points')
# plt.plot(traveler_count_line, latency_line, color='red', label=f'Best fit: y={beta[e]:.2f}x+{alpha_mean_nominal[e]:.2f}')

# plt.xlabel("Traveler Count")
# plt.ylabel("Latency")
# plt.title("Least Squares Best-Fit Line")
# plt.legend()
# plt.grid(True)

# plt.show()



In [108]:
B = np.diag(beta)
edge_node_matrix = np.array([[0, 2], [0, 1], [2, 6], [2, 3], [1, 3], \
                             [1, 4], [3, 5], [4, 6], [4, 5], [5, 6]])

delta = 1.0
eta = 2.0

R = generate_R_from_edge_node_matrix(edge_node_matrix)
Gamma = generate_Gamma(B, R)
eps_max = compute_eps_max(Gamma, B, R, alpha_mean_nominal, eta, delta)
eps_max_adjusted = min(eps_max, min(alpha_mean_nominal))

print("eps_max_adjusted:", eps_max_adjusted)

# Compute distributionally robust toll and corresponding worst-case latency (in theory):

eps = 0.5 * eps_max_adjusted
tau_opt, worst_case_ = compute_opt_DR_toll(Gamma, B, R, alpha_mean_nominal, eta, eps, delta)

# Compute certainty equivalence toll and corresponding worst-case latency (in theory):



# def generate_R_from_edge_node_matrix(edge_node_matrix):
#     return R

# def generate_Gamma(B, R):
#     return inv_B - inv_B @ R.T @ R_inv_B_R_top @ R @ inv_B

# def compute_eps_max(Gamma, B, R, alpha_mean_nominal, eta, delta):
#     return eps.value
    
# def compute_opt_DR_toll(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
#     return tau.value, prob.value   




Gamma.shape (10, 10)
alpha_mean_nominal.shape: (10,)


eps_max: 1.5308301970013645

eps_max_adjusted: [1.5308302]

DR optimal toll: [-0.          3.98871447 36.7541289   6.66307277  8.77905576  6.37020047
 10.68969074 25.6389507   4.61358925  4.84915787]

Worst-case DR latency: 7.882184567955359



# <font color='red'> Red topic

# Scratch Work

In [6]:
# Generate a random non-trivial linear program.
m = 15
n = 10
np.random.seed(1)
s0 = np.random.randn(m)
lamb0 = np.maximum(-s0, 0)
s0 = np.maximum(s0, 0)
x0 = np.random.randn(n)
A = np.random.randn(m, n)
b = A @ x0 + s0
c = -A.T @ lamb0

# Define and solve the CVXPY problem.
x = cp.Variable(n)
obj = cp.Minimize(c.T@x)
constr = [A @ x <= b]
prob = cp.Problem(obj, constr)
prob.solve()

# Print result.
print("\nThe optimal value is", prob.value)
print("A solution x is")
print(x.value)
print("A dual solution is")
print(prob.constraints[0].dual_value)




The optimal value is -15.22091260446552
A solution x is
[-1.09523662 -0.20091053 -0.81496706  0.07463463  0.50506136 -1.01590466
  1.19326977  0.96555774  0.5403823   0.90851532]
A dual solution is
[8.48822518e-14 6.11756414e-01 5.28171752e-01 1.07296862e+00
 2.39970175e-13 2.30153870e+00 1.03698132e-13 7.61206901e-01
 3.71791797e-13 2.49370375e-01 1.01626544e-13 2.06014071e+00
 3.22417204e-01 3.84054355e-01 3.65690265e-14]
